# Labwork 4 - Second-order methods and the statistics of the loss

**Week 2 - Day 4 - ~ 165 min at the keyboard**

Read Lecture 4 first. Today you use the **curvature**: a Cholesky solver, Newton, damping,
and the penalty that makes a loss well posed. Each exercise is followed by the figures that
measure what you just wrote.

> **The rule.** `interfaces.py`, `results.py`, `errors.py` and `types.py` are **provided** -
> never edit them. Everywhere else, replace each `raise NotImplementedError`, keeping the
> signature and honouring the docstring.

## Setup

Run this once. It points the notebook at your `optlab` clone, turns on `%autoreload` so
that the edits you make in `src/optlab/` are picked up here without restarting the kernel,
and gives you a `check()` helper that runs a test file and reports what happened.

`check()` only trims pytest's output so a failure stays readable. Anything else, run
directly with Jupyter's `!` escape - these are the real commands, and the ones you will
type in a terminal for the rest of your life:

```
!cd {OPTLAB} && make check          # the whole thing: tests + mypy + ruff
!cd {OPTLAB} && python -m mypy      # types only
!cd {OPTLAB} && python -m pytest -q -m day1
```

`!` runs each line in its own shell, so the `cd` has to sit on the same line as the
command.

In [2]:
%load_ext autoreload
%autoreload 2

import os
import subprocess
import sys
from pathlib import Path

# Adjust if your clone lives elsewhere.
OPTLAB = (Path.cwd() / ".." / ".." / "optlab").resolve()
assert OPTLAB.exists(), f"optlab not found at {OPTLAB} -- edit OPTLAB above"
if not (OPTLAB / "src" / "optlab" / "interfaces.py").exists():
    # Not a student clone. The teacher tree keeps only the solved files in src/ and
    # assembles a complete package in code/ (`make check`), so use that one. This branch
    # never fires for you; it is here so the correction notebook can be a plain copy of
    # this one instead of a second version that drifts.
    OPTLAB = OPTLAB / "code"
    assert OPTLAB.exists(), f"run `make check` in {OPTLAB.parent} first"
ROOT = OPTLAB
print("optlab:", OPTLAB)

# Prefer the installed package (`make install`); fall back to <root>/src if it is not.
try:
    import optlab
except ModuleNotFoundError:
    sys.path.insert(0, str(OPTLAB / "src"))
    import optlab
sys.path.insert(0, str(OPTLAB))  # for datasets/


def check(*pytest_args):
    """Run pytest inside the optlab clone and show a short report."""
    # pytest runs in a SUBPROCESS, and a subprocess does not inherit the sys.path entries
    # added above. Passing them through PYTHONPATH is what makes `check()` work before
    # `make install` has been run, instead of failing to import optlab in conftest.py.
    env = {**os.environ, "PYTHONPATH": os.pathsep.join(
        [str(OPTLAB / "src"), str(OPTLAB), os.environ.get("PYTHONPATH", "")]).rstrip(os.pathsep)}
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", "--no-header", *pytest_args],
                       cwd=OPTLAB, capture_output=True, text=True, env=env)
    out = r.stdout + r.stderr
    tail = [l for l in out.splitlines() if l.strip()][-12:]
    print("\n".join(tail))
    if "No module named pytest" in out:
        verdict = "pytest is not installed -- run:  pip install -e '.[dev]'"
    elif r.returncode == 0:
        verdict = "PASSED"
    elif r.returncode == 5:
        # Exit code 5 means pytest collected nothing at all. That is NOT a failure of
        # your code: no test in the suite matches what was asked for. Some days have no
        # automated tests yet; judge those exercises by the checks written in the text.
        verdict = "no tests matched -- nothing to run here, this is not a failure"
    else:
        verdict = "not yet -- keep going"
    print()
    print(verdict)


def edit(relpath):
    """Print the absolute path of a source file, so you can open it in the editor."""
    print(OPTLAB / "src" / "optlab" / relpath)


check("tests/test_no_oracle_in_src.py")   # provided, and already green

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
optlab: /Users/jp/Documents/USTH_materials/Master_Course_Materials/BC1.01_Programming_Techniques/Week2/optlab
..............................                                           [100%]
30 passed in 0.07s
In /Users/jp/Documents/USTH_materials/Master_Course_Materials/BC1.01_Programming_Techniques/Week2/.venv/lib/python3.15/site-packages/__editable__.optlab-0.1.0.pth: /Users/jp/Documents/USTH materials/Master_Course_Materials/BC1.01_Programming_Techniques/Week2/optlab/src does not exist; skipping sys.path append

PASSED


In [3]:
import matplotlib.pyplot as plt
import numpy as np

np.set_printoptions(precision=6, suppress=True)
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.3, "font.size": 10,
                     "axes.titlesize": 10, "figure.dpi": 110})
rng = np.random.default_rng(20250921)

print("optlab root  :", ROOT)
print("optlab loaded:", Path(optlab.__file__).parent)

optlab root  : /Users/jp/Documents/USTH_materials/Master_Course_Materials/BC1.01_Programming_Techniques/Week2/optlab
optlab loaded: /Users/jp/Documents/USTH_materials/Master_Course_Materials/BC1.01_Programming_Techniques/Week2/optlab/src/optlab


In [4]:
def status(name, thunk):
    """Report whether one piece of the package is implemented, without a traceback."""
    try:
        thunk()
    except NotImplementedError:
        return f"  MISSING   {name}"
    except Exception as err:                      # noqa: BLE001 - we want to see anything
        return f"  BROKEN    {name}   ({type(err).__name__}: {err})"
    return f"  ok        {name}"

In [5]:
from optlab.errors import NotPositiveDefiniteError
from optlab.linalg import CholeskySolver, cholesky
from optlab.linesearch import Armijo
from optlab.objective_ops import RegularizedObjective
from optlab.observers import History
from optlab.optimizers import (DescentOptimizer, NewtonDirection, SteepestDescent,
                               newton)
from optlab.problems import Quadratic, Rosenbrock, logistic_regression
from optlab.regularizers import L2, NoRegularizer
from optlab.stopping import AnyOf, GradientNormBelow, MaxIterations

_A = np.array([[4.0, 1.0], [1.0, 3.0]])
_q = Quadratic(_A, np.array([1.0, -2.0]))
_w = np.zeros(2)

print("Day 4 readiness")
print(status("Quadratic.hessian", lambda: _q.hessian(_w)))
print(status("cholesky", lambda: cholesky(_A)))
print(status("CholeskySolver", lambda: CholeskySolver().solve(_A, np.ones(2))))
print(status("NewtonDirection",
             lambda: NewtonDirection(CholeskySolver()).direction(_q, _w, _q.gradient(_w))))
print(status("newton", lambda: newton(max_iter=3).minimize(_q, _w)))
print(status("NoRegularizer", lambda: NoRegularizer().value(_w)))
print(status("L2", lambda: L2(1.0).value(_w)))
print(status("RegularizedObjective",
             lambda: RegularizedObjective(_q, L2(1.0)).hessian(_w)))
print()
print("Also needed here")
print(status("GLMLoss.hessian",
             lambda: logistic_regression(np.eye(2), np.array([1.0, 0.0])).hessian(_w)))
print(status("Rosenbrock.hessian", lambda: Rosenbrock().hessian(_w)))

Day 4 readiness
  MISSING   Quadratic.hessian
  MISSING   cholesky
  MISSING   CholeskySolver
  MISSING   NewtonDirection
  MISSING   newton
  MISSING   NoRegularizer
  MISSING   L2
  MISSING   RegularizedObjective

Also needed here
  MISSING   GLMLoss.hessian
  ok        Rosenbrock.hessian


---

## Exercise 1 - Cholesky  *(~ 55 min)*

Implement `cholesky`, `solve_lower`, `solve_upper_from_lower` and `CholeskySolver`.

Vectorize the inner loop over `i` - three nested Python loops will be unusably slow on the
day-6 benchmark.

**Raise `NotPositiveDefiniteError` as soon as a pivot is non-positive.** Do not clamp it,
do not add a small epsilon, do not fall back to `np.linalg.solve`. The failure carries
information and the caller needs to see it.

Three checks: the hand example `[[4,2,2],[2,5,3],[2,3,6]] -> L = [[2,0,0],[1,2,0],[1,1,2]]`;
agreement with `np.linalg.solve` on random SPD matrices; and the refusal on
`[[1,2],[2,1]]`, whose second pivot would need `sqrt(-3)`.

Then try a Hilbert matrix. The residual `||Ax - b||` will be tiny while the error
`||x - x_true||` is enormous - a small residual does **not** mean an accurate answer.

**Open:** `src/optlab/linalg/cholesky.py`

In [10]:
edit("linalg/cholesky.py")
check("tests/unit/test_day4_cholesky.py", "tests/contracts/test_linear_solver_contract.py")

/Users/jp/Documents/USTH_materials/Master_Course_Materials/BC1.01_Programming_Techniques/Week2/optlab/src/optlab/linalg/cholesky.py
       [2., 1.]]), b = array([1., 1.])
    def solve(self, A: Mat, b: Vec) -> Vec:
>       raise NotImplementedError("[DAY 4] lab 1")
E       NotImplementedError: [DAY 4] lab 1
src/optlab/linalg/cholesky.py:58: NotImplementedError
=========================== short test summary info ============================
FAILED tests/unit/test_day4_cholesky.py::test_triangular_solves - NotImplementedError: [DAY 4] lab 1
FAILED tests/contracts/test_linear_solver_contract.py::test_solves_an_spd_system[cholesky] - NotImplementedError: [DAY 4] lab 1
FAILED tests/contracts/test_linear_solver_contract.py::test_matches_the_numpy_oracle[cholesky] - NotImplementedError: [DAY 4] lab 1
FAILED tests/contracts/test_linear_solver_contract.py::test_rejects_a_non_positive_definite_matrix[cholesky] - NotImplementedError: [DAY 4] lab 1
4 failed, 4 passed in 0.13s
In /Users/jp/Document

<details>
<summary><b>Plan B</b> - open only if you are stuck for more than ten minutes</summary>

`L[j+1:, j] = (A[j+1:, j] - L[j+1:, :j] @ L[j, :j]) / L[j, j]` does the whole column at once. Forward substitution: `y[i] = (b[i] - L[i, :i] @ y[:i]) / L[i, i]`.

</details>

---

## Exercise 2 - Hessians and Newton  *(~ 50 min)*

Implement the three `hessian` methods, then `NewtonDirection` and the `newton()` assembly.

`check_hessian` is **provided**, and it is worth thirty seconds of reading: it contains no
finite-difference routine at all, it hands your *gradient* to day 1's
`numerical_jacobian`, because a Hessian **is** the Jacobian of the gradient. That reuse is
the design point; writing the same five lines a second time is not.

`GLMLoss.hessian` is `X^T DX / n` with `D = diag(phi''(Xw, y))`. Get that as one vectorized
expression, not a loop over samples.

`NewtonDirection` receives its solver by constructor and calls `solve(H, -g)`. Let
`NotPositiveDefiniteError` propagate - Exercise 3 deals with it.

Four things to verify:

1. **Linear regression converges in exactly one iteration**, matching `np.linalg.solve` on
   the normal equations.
2. Logistic regression reaches the same minimizer as your day-2 gradient descent.
3. The error ratio `e_{k+1}/e_k^2` stays bounded - that is quadratic convergence.
4. **You did not edit `descent.py`.** `newton()` is four existing objects wired together.

**Open:** `src/optlab/numerics/gradcheck.py`, `problems/*.py`, `optimizers/directions.py`, `optimizers/descent.py`

In [7]:
check("-m", "day4")

import subprocess
d = subprocess.run(["git", "diff", "--stat", "src/optlab/optimizers/descent.py"],
                   cwd=OPTLAB, capture_output=True, text=True).stdout.strip()
print("\nchanges to the day-2 loop:", d if d else "none -- Newton needed no new loop")

FAILED tests/unit/test_day4_newton.py::test_modified_newton_returns_a_descent_direction_on_an_indefinite_hessian - NotImplementedError: [DAY 4] lab 3
FAILED tests/unit/test_day4_newton.py::test_the_damping_does_not_persist_between_calls - NotImplementedError: [DAY 4] lab 3
FAILED tests/unit/test_day4_newton.py::test_modified_newton_gives_up_rather_than_lying - NotImplementedError: [DAY 4] lab 3
FAILED tests/unit/test_day4_newton.py::test_no_regularizer_is_the_neutral_element - NotImplementedError: [DAY 4] lab 4
FAILED tests/unit/test_day4_newton.py::test_l2_value_gradient_and_prox - NotImplementedError: [DAY 4] lab 4
FAILED tests/unit/test_day4_newton.py::test_regularized_objective_adds_the_two_parts - NotImplementedError: [DAY 4] lab 4
FAILED tests/unit/test_day4_newton.py::test_the_regularized_gradient_is_the_gradient_of_the_regularized_value - NotImplementedError: [DAY 4] lab 4
FAILED tests/unit/test_day4_newton.py::test_ridge_lifts_every_eigenvalue_by_lambda - NotImplementedError: 

<details>
<summary><b>Plan B</b> - open only if you are stuck for more than ten minutes</summary>

For the GLM Hessian: `X.T @ (X * d2[:, None]) / n`. Broadcasting scales each row of `X` by its weight, which is exactly `X^T DX` without ever forming `D`.

</details>

## 1. On a quadratic, Newton is not an iterative method

If $f$ is exactly quadratic then the model is exactly $f$, and solving $Hd = -g$ lands on
the minimiser. Not close to it - on it. Your `Quadratic` is
$f(x) = \tfrac{1}{2}x^\top A x - b^\top x$, so the minimiser is $A^{-1}b$, and we can
check the claim against `np.linalg.solve`.

In [8]:
A = np.array([[4.0, 1.0], [1.0, 3.0]])
b_vec = np.array([1.0, -2.0])
quad = Quadratic(A, b_vec)
x0 = np.array([5.0, -7.0])

res = newton(max_iter=20).minimize(quad, x0)
print(f"started at        {x0}")
print(f"Newton result     {res.x}   after {res.iterations} iteration(s)")
print(f"np.linalg.solve   {np.linalg.solve(A, b_vec)}")
print(f"gradient norm     {res.grad_norm:.3e}")
print(f"converged         {res.converged}")

NotImplementedError: [DAY 4] lab 2

One iteration, gradient norm $1.8\times10^{-15}$ - the residual of a linear solve, not the
tail of a convergence. Everything after this section is about what happens when the
quadratic model is only an *approximation*.

> **A useful way to hold it.** Gradient descent asks "which way is downhill?". Newton
> asks "where is the bottom of the bowl that best matches $f$ here?" and goes straight
> there. When $f$ *is* a bowl, that is the answer. When $f$ is not, it is a guess - and
> Sec. 3 is about how badly that guess can fail.

## 2. Quadratic convergence: the error squares every step

On a smooth function near a minimum with $H \succ 0$, Newton satisfies

$$\|x_{k+1} - x^*\| \;\le\; C\,\|x_k - x^*\|^2 .$$

Squaring the error means *doubling the correct digits* at every iteration. That is a
qualitatively different animal from the constant factor $(\kappa-1)/(\kappa+1)$ of
Labwork 2, and it is why the iteration counts below are not a typo.

Same logistic regression for both methods, same Armijo line search, same starting point.

In [ ]:
n, d = 500, 6
data_rng = np.random.default_rng(11)
X = data_rng.normal(size=(n, d))
w_gen = data_rng.normal(size=d) * 1.5
y = (data_rng.random(n) < 1.0 / (1.0 + np.exp(-(X @ w_gen)))).astype(float)

loss = logistic_regression(X, y)
w0 = np.zeros(d)


def trace(direction, problem, start, tol=1e-12, max_iter=50):
    """Run one DescentOptimizer and return its gradient norms, one per iteration."""
    hist = History()
    DescentOptimizer(direction, Armijo(),
                     AnyOf(GradientNormBelow(tol), MaxIterations(max_iter)),
                     observers=[hist]).minimize(problem, np.asarray(start, dtype=float))
    return np.array([e.grad_norm for e in hist.events])


g_newton = trace(NewtonDirection(CholeskySolver()), loss, w0)
g_descent = trace(SteepestDescent(), loss, w0, max_iter=20000)

print(f"Newton stopped after {len(g_newton)} iterations, |g| = {g_newton[-1]:.3e}")
print(f"GD ran {len(g_descent)} iterations, |g| = {g_descent[-1]:.3e}")
print()
print(f"{'target |g|':>12} {'Newton':>10} {'gradient descent':>18}")
for tol in (1e-2, 1e-4, 1e-6, 1e-8, 1e-10):
    hit_n = np.argmax(g_newton <= tol) + 1 if (g_newton <= tol).any() else None
    hit_g = np.argmax(g_descent <= tol) + 1 if (g_descent <= tol).any() else None
    print(f"{tol:12.0e} {str(hit_n):>10} {str(hit_g):>18}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].semilogy(np.arange(1, len(g_newton) + 1), g_newton, "o-", label="Newton")
ax[0].semilogy(np.arange(1, len(g_descent) + 1), g_descent, "-", lw=1,
               label="gradient descent")
ax[0].set_xlim(0, 60)
ax[0].set_xlabel("iteration")
ax[0].set_ylabel(r"$\|\nabla f\|$")
ax[0].set_title("The first 60 iterations")
ax[0].legend(fontsize=8)

logs = np.log10(g_newton)
ax[1].plot(logs[:-1], logs[1:], "o", ms=7, label="Newton steps")
keep = g_newton[1:] > 1e-15                      # drop the float64-limited final step
fit_all = np.polyfit(logs[:-1], logs[1:], 1)
fit_keep = np.polyfit(logs[:-1][keep], logs[1:][keep], 1)
span = np.array([logs[:-1].min(), logs[:-1].max()])
ax[1].plot(span, np.polyval(fit_keep, span), "k-", lw=1,
           label=f"fit, last step dropped: slope {fit_keep[0]:.2f}")
ax[1].plot(span, np.polyval(fit_all, span), "r--", lw=1,
           label=f"fit, all points: slope {fit_all[0]:.2f}")
ax[1].plot(logs[:-1][~keep], logs[1:][~keep], "rx", ms=11,
           label=r"limited by float64, not by Newton")
ax[1].set_xlabel(r"$\log_{10}\|\nabla f(x_k)\|$")
ax[1].set_ylabel(r"$\log_{10}\|\nabla f(x_{k+1})\|$")
ax[1].set_title("Slope 2 would be exact quadratic convergence")
ax[1].legend(fontsize=7, loc="upper left")

fig.suptitle("Figure 1 - Newton vs gradient descent on the same logistic regression")
fig.tight_layout()
plt.show()

**Left: Newton's points fall off the bottom of the plot** while gradient descent is still
on its first decade. Read the table for the numbers: to reach $\|g\| \le 10^{-8}$, Newton
needs 6 iterations and gradient descent 568. At $10^{-10}$ gradient descent does not get
there at all within its budget.

**Right - read this one carefully.** Quadratic convergence predicts that $\log\|g_{k+1}\|$
against $\log\|g_k\|$ has slope 2. The fit over *all* points gives about 1.55; with the
final step dropped it gives 1.91.

That final step lands at $\|g\| \approx 3\times10^{-17}$ - the size of the rounding
error, not of anything Newton computed. The theory is confirmed to the limit of the
arithmetic, and then the arithmetic runs out.

> This is the general shape of validating a fast method: the last few iterations measure
> your floating-point format, not your algorithm.

---

## Exercise 3 - Damping, and a failure worth seeing  *(~ 30 min)*

Newton does not look for a minimum. It looks for `grad f = 0`, and a maximum satisfies that
just as well.

**1. See it.** On the double well `f(x) = (x^2 - 1)^2`, start at `x_0 = 0.3` where
`f''(0.3) < 0` and write the scalar iteration `x <- x - f'(x)/f''(x)` by hand - three lines
of numpy, no optlab. You land on the maximum `x = 0` in about three steps and stay there.

**2. Now through your own code.** `NewtonDirection(CholeskySolver())` with `FixedStep(1.0)`
does *not* reproduce that walk: it raises `NotPositiveDefiniteError` on the first
iteration. Your Exercise 1 refusal, working as designed.

**3. Try to repair it with `Armijo()`.** It raises the same error. A line search chooses
how *far* to go along a direction; it cannot help when no direction was produced. Be able
to state that distinction - it is why the next repair lives in the `IDirectionRule`.

**4. Fix the direction.** Fill in `ModifiedNewton`: catch `NotPositiveDefiniteError`, solve
`(H + tau*I)d = -g`, double `tau` until Cholesky succeeds. Read its docstring first. Then
check, with a `History`, that from `x_0 = 0.3` it reaches `x = +1`, that the recorded
values are *strictly* decreasing, and that damping is active for a step or two and then
gets out of the way.

**5. When is `Armijo` the right repair?** When the Hessian *is* positive definite but the
full step overshoots. Run Rosenbrock from `x_0 = (-1.2, 1)` with `FixedStep(1.0)` and then
with `Armijo()`. Do not compare iteration counts - the unit step wins there. Compare the
paths: the unit step throws the iterate out to `f = 1.4e3` before coming back, `Armijo`
never lets `f` rise at all.

Two failures, two fixes: an indefinite Hessian is a *direction* problem, an overshooting
step is a *length* problem. `H + tau*I` comes back tomorrow as Levenberg-Marquardt.

**Open:** `src/optlab/optimizers/directions.py`

In [ ]:
check("-m", "day4")

## 3. When Cholesky refuses, and why that is the right behaviour

$Hd = -g$ has a solution whenever $H$ is invertible. But the solution is a *descent*
direction only when $H$ is **positive definite**; if $H$ has a negative eigenvalue, the
quadratic model has a direction that curves downward forever, and its "minimum" is a
saddle or a maximum. Newton will walk towards it.

Cholesky exists precisely to detect this. $H = LL^\top$ requires taking square roots of
the pivots, and on a non-positive-definite matrix a pivot goes negative and there is no
real root. Your `CholeskySolver` raises `NotPositiveDefiniteError` there. The stub
docstring insists you **propagate** it rather than quietly falling back to $-g$, and this
section is why: the exception is information about the problem, not a nuisance.

Rosenbrock has an indefinite region, and it is easy to find analytically. With
$H = \begin{pmatrix} 2 - 400(y - x^2) + 800x^2 & -400x \\ -400x & 200\end{pmatrix}$,
$\det H = 400 - 80000\,y + 80000\,x^2$, so $H$ fails to be positive definite exactly
where $y > x^2 + 1/200$. Let us have your code confirm that.

In [ ]:
ros = Rosenbrock()
grid = 220
xs = np.linspace(-2.0, 2.0, grid)
ys = np.linspace(-1.0, 3.0, grid)
XX, YY = np.meshgrid(xs, ys)

indefinite = np.zeros_like(XX, dtype=bool)
values = np.zeros_like(XX)
for i in range(grid):
    for j in range(grid):
        p = np.array([XX[i, j], YY[i, j]])
        values[i, j] = ros.value(p)
        try:
            cholesky(ros.hessian(p))
        except NotPositiveDefiniteError:
            indefinite[i, j] = True

predicted = YY > XX ** 2 + 1.0 / 200.0
agree = np.mean(indefinite == predicted)
print(f"grid points where Cholesky refuses : {indefinite.sum()} of {indefinite.size}")
print(f"agreement with the region y > x^2 + 1/200 : {agree * 100:.3f}%")
print(f"disagreeing points (all on the boundary)  : {np.sum(indefinite != predicted)}")

In [ ]:
starts = [(-1.2, 1.0), (2.0, -1.0), (0.0, 2.0), (-0.5, 0.5)]
outcomes = {}

for s in starts:
    hist = History()
    try:
        r = DescentOptimizer(NewtonDirection(CholeskySolver()), Armijo(),
                             AnyOf(GradientNormBelow(1e-10), MaxIterations(60)),
                             observers=[hist]).minimize(ros, np.array(s, dtype=float))
        outcomes[s] = ("converged", r.iterations, np.vstack([s] + [e.x for e in hist.events]))
        print(f"from {s}: converged to {np.round(r.x, 6)} in {r.iterations} iterations")
    except NotPositiveDefiniteError:
        path = np.vstack([s] + [e.x for e in hist.events])
        outcomes[s] = ("refused", len(hist.events), path)
        print(f"from {s}: NotPositiveDefiniteError after {len(hist.events)} accepted step(s)")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 6))

ax.contour(XX, YY, np.log10(values + 1e-8), levels=25, colors="0.75", linewidths=0.6)
ax.contourf(XX, YY, indefinite.astype(float), levels=[0.5, 1.5],
            colors=["tab:red"], alpha=0.16)
ax.plot(xs, xs ** 2 + 1.0 / 200.0, "r-", lw=1.2,
        label=r"boundary $y = x^2 + 1/200$")

for s, (what, k, path) in outcomes.items():
    colour = "tab:green" if what == "converged" else "tab:red"
    ax.plot(path[:, 0], path[:, 1], "o-", color=colour, ms=3.5, lw=1.1)
    ax.annotate(f"{s}\n{what}", s, textcoords="offset points", xytext=(6, 6),
                fontsize=7.5, color=colour)

ax.plot(1.0, 1.0, "k*", ms=13, label="minimum $(1, 1)$")
ax.set_xlim(-2.15, 2.45)
ax.set_ylim(-1.35, 3.05)
ax.set_xlabel("$x$")
ax.set_ylabel("$y$")
ax.set_title("Figure 2 - Rosenbrock: shaded where the Hessian is not positive definite\n"
             "(green: Newton finished, red: Cholesky refused)")
ax.legend(fontsize=8, loc="lower right")
fig.tight_layout()
plt.show()

**The shaded region is where your `cholesky` raises**, and it agrees with
$y > x^2 + 1/200$ on **every one** of the 48 400 grid points - a stronger test of
`cholesky` than any unit test in the suite.

Now the four starting points. Two are outside the region and Newton finishes. Two are
inside and the very first call raises, after **0** accepted steps: there is no partial
progress to salvage.

Follow the path from $(-1.2, 1.0)$: it runs along the *underside* of the boundary for its
whole length. Rosenbrock's valley lies just outside the indefinite region, so a method that
tracks the valley is permanently one bad step away from a Hessian it cannot factorise.
That is why damping is not optional - and note that the classic starting point sits just
below the parabola, so the textbook demo never meets this failure.

## 4. What a Newton step costs

Iteration counts are not run times. A gradient descent step on a GLM costs $O(nd)$; a
Newton step additionally forms $X^\top D X$ at $O(nd^2)$ and factorises it at $O(d^3)$.
So Newton is a bargain at $d = 10$ and unusable at $d = 10^6$, and the crossover is
somewhere you should measure rather than guess.

We time the `direction` call itself, which is the honest unit: for `SteepestDescent` that
is a gradient, for `NewtonDirection` it is a gradient, a Hessian and a Cholesky solve.

In [ ]:
import time

dims = [8, 16, 32, 64, 128, 256]
n_fixed = 5000
t_gd, t_newton = [], []

sd = SteepestDescent()
nd = NewtonDirection(CholeskySolver())

print(f"{'d':>5} {'GD step (ms)':>14} {'Newton step (ms)':>18} {'ratio':>8}")
for dim in dims:
    gen = np.random.default_rng(0)
    Xd = gen.normal(size=(n_fixed, dim))
    yd = (gen.random(n_fixed) < 1.0 / (1.0 + np.exp(-(Xd @ gen.normal(size=dim))))).astype(float)
    ld = logistic_regression(Xd, yd)
    wd = np.zeros(dim)

    reps = 30
    t0 = time.perf_counter()
    for _ in range(reps):
        sd.direction(ld, wd, ld.gradient(wd))
    a = (time.perf_counter() - t0) / reps

    t0 = time.perf_counter()
    for _ in range(reps):
        nd.direction(ld, wd, ld.gradient(wd))
    c = (time.perf_counter() - t0) / reps

    t_gd.append(a)
    t_newton.append(c)
    print(f"{dim:5d} {a * 1e3:14.3f} {c * 1e3:18.3f} {c / a:8.2f}")

s_gd = np.polyfit(np.log(dims), np.log(t_gd), 1)[0]
s_nt = np.polyfit(np.log(dims), np.log(t_newton), 1)[0]
print(f"\nmeasured log-log slopes:  GD {s_gd:.2f}   Newton {s_nt:.2f}")
print(f"(n is fixed at {n_fixed}, so the theoretical slopes are 1 and up to 3)")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].loglog(dims, np.array(t_gd) * 1e3, "o-", label=f"gradient step, slope {s_gd:.2f}")
ax[0].loglog(dims, np.array(t_newton) * 1e3, "s-", label=f"Newton step, slope {s_nt:.2f}")
ax[0].set_xlabel("dimension $d$")
ax[0].set_ylabel("time per direction (ms)")
ax[0].set_title(f"Cost of one step ($n = {n_fixed}$ fixed)")
ax[0].legend(fontsize=8)

ax[1].semilogx(dims, np.array(t_newton) / np.array(t_gd), "o-")
ax[1].set_xlabel("dimension $d$")
ax[1].set_ylabel("Newton step / GD step")
ax[1].set_title("How many gradient steps a Newton step is worth")
ax[1].axhline(568 / 6, color="k", ls="--", lw=1,
              label="568/6: the Sec. 2 iteration ratio")
ax[1].legend(fontsize=8)

fig.suptitle("Figure 3 - iterations are cheap for Newton, steps are not")
fig.tight_layout()
plt.show()

**These are the only numbers here that depend on your machine**, so read the shape rather
than the digits.

The dashed line is the iteration advantage Newton had above. While the cost ratio stays
*below* it, Newton wins on wall-clock time. Where your curve crosses is where Newton stops
paying for itself on your hardware.

Now the slopes in the left legend. Against $O(nd)$ and $O(nd^2 + d^3)$ you would expect 1
and up to 3; the measured values are far lower. Not an error in the theory - at these sizes
a tuned BLAS is nowhere near its asymptotic regime, and Python call overhead dominates the
gradient outright. **Complexity exponents describe a limit, and the limit is further out
than people assume**, which is why the answer is to time it rather than to reason about
exponents.

> **The practical rule.** Newton for $d$ in the tens or low hundreds with $n$ moderate -
> which covers most classical statistics. First-order or quasi-Newton methods beyond that.

---

## Exercise 4 - Ridge, for free  *(~ 30 min)*

Implement `NoRegularizer` and `L2` (`value = 1/2*lam||w||^2`, `gradient = lam*w`,
`prox = w/(1+lam*t)`), then `RegularizedObjective` - an adapter presenting `f + r` as a single
`IObjective`.

Now the payoff. Without writing a single new optimizer:

```python
ridge = RegularizedObjective(GLMLoss(X, y, SquaredError()), L2(lam))
newton().minimize(ridge, w0)          # ridge regression
gradient_descent().minimize(ridge, w0)  # also ridge regression
```

Every optimizer you have written gained a regularized version the moment this adapter
existed. Check the Newton solution against the closed form `(X^T X/n + lam*I)^-1 X^T y/n`.

Then look at the eigenvalues of `H` and `H + lam*I`: every one lifted by exactly `lam`. That is
why ridge can never break Cholesky, and it is the last piece of day 5.

**Open:** `src/optlab/regularizers.py`, `src/optlab/objective_ops.py`

In [ ]:
check("-m", "day4")

## 5. The problem Newton cannot fix: an optimum that does not exist

Here is a failure that has nothing to do with the algorithm. Take logistic regression on
data that a hyperplane separates **perfectly**. For any $w$ that classifies everything
correctly, doubling $w$ makes every predicted probability more confident and strictly
lowers the loss. So $f(2w) < f(w)$, always, and

$$\inf_w f(w) = 0 \text{ is never attained: the maximum-likelihood estimate does not exist.}$$

Watch a *correct* implementation of Newton chase it.

In [ ]:
n_sep, d_sep = 80, 6
sep_rng = np.random.default_rng(5)
X_sep = sep_rng.normal(size=(n_sep, d_sep))
w_sep = np.array([1.5, -1.0, 0.8, 0.0, -0.4, 0.2])
y_sep = (X_sep @ w_sep > 0).astype(float)       # no noise, so separable by construction

separable = logistic_regression(X_sep, y_sep)
w0_sep = np.zeros(d_sep)
print(f"smallest margin in the data: {np.min((2 * y_sep - 1) * (X_sep @ w_sep)):.4f}  "
      "(> 0 means perfectly separable)")

print(f"\n{'budget':>8} {'||w||':>12} {'f(w)':>14} {'||grad||':>12}")
for budget in (5, 10, 20, 40):
    r = DescentOptimizer(NewtonDirection(CholeskySolver()), Armijo(),
                         MaxIterations(budget)).minimize(separable, w0_sep)
    print(f"{budget:8d} {np.linalg.norm(r.x):12.3f} {r.value:14.4e} {r.grad_norm:12.2e}")

**Every column is behaving correctly** - $f$ and $\|g\|$ both fall to around $10^{-17}$,
and a stopping rule based on either would report success. Meanwhile $\|w\|$ has gone
$14.5 \to 59 \to 170 \to 395$ and is still climbing.

Notice what you do **not** see: no overflow warning, even though $|Xw|$ is in the hundreds -
the range that broke the naive sigmoid on day 1. Your guarded `softplus` and `sigmoid`
exponentiate only $-|z|$, so they underflow quietly instead. Day 1's stability work is what
keeps this cell honest; the evidence is the $\|w\|$ column, and nothing else was ever
going to raise.

This is the failure a gradient-norm criterion cannot see, and it is common: it happens
whenever a feature perfectly predicts the outcome - a leaked label, a duplicated column,
more features than samples.

The fix is not a better optimizer. It is a different objective: penalise $\|w\|$ so that
running away stops being free.

In [ ]:
lams = np.logspace(1, -4, 12)
paths, norms, kappas, iters, converged = [], [], [], [], []

for lam in lams:
    ridge = RegularizedObjective(separable, L2(lam))
    r = newton(tol=1e-10, max_iter=200).minimize(ridge, w0_sep)
    ev = np.linalg.eigvalsh(ridge.hessian(r.x))
    paths.append(r.x)
    norms.append(np.linalg.norm(r.x))
    kappas.append(ev[-1] / ev[0])
    iters.append(r.iterations)
    converged.append(r.converged)

paths = np.array(paths)
print(f"{'lambda':>10} {'||w||':>10} {'kappa(H)':>12} {'iters':>7} {'converged':>10}")
for i, lam in enumerate(lams):
    print(f"{lam:10.2e} {norms[i]:10.4f} {kappas[i]:12.3e} {iters[i]:7d} "
          f"{str(converged[i]):>10}")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 4))

for j in range(d_sep):
    ax[0].semilogx(lams, paths[:, j], "o-", ms=3,
                   label=f"$w_{j + 1}$ (true {w_sep[j]:+.1f})")
ax[0].axhline(0.0, color="k", lw=0.8)
ax[0].invert_xaxis()
ax[0].set_xlabel(r"$\lambda$ (decreasing $\rightarrow$)")
ax[0].set_ylabel("coefficient")
ax[0].set_title("The ridge path")
ax[0].legend(fontsize=6.5, ncol=2)

ax[1].loglog(lams, norms, "o-")
ax[1].invert_xaxis()
ax[1].set_xlabel(r"$\lambda$ (decreasing $\rightarrow$)")
ax[1].set_ylabel(r"$\|w\|$")
ax[1].set_title(r"$\|w\|$ is finite for every $\lambda > 0$," "\n" r"and diverges as $\lambda \to 0$")

ax[2].loglog(lams, kappas, "o-", label=r"$\kappa(H)$ at the solution")
ax2 = ax[2].twinx()
ax2.semilogx(lams, iters, "s--", color="tab:orange", label="Newton iterations")
ax2.set_ylabel("Newton iterations", color="tab:orange")
ax[2].invert_xaxis()          # twinx shares the axis; inverting both would undo it
ax[2].set_xlabel(r"$\lambda$ (decreasing $\rightarrow$)")
ax[2].set_ylabel(r"$\kappa(H)$")
ax[2].set_title("The same $\\lambda I$ also conditions the Hessian")
ax[2].legend(fontsize=8, loc="upper left")
ax2.legend(fontsize=8, loc="lower right")

fig.suptitle("Figure 4 - ridge on perfectly separable data")
fig.tight_layout()
plt.show()

**Left.** Every coefficient shrinks smoothly towards zero as $\lambda$ grows. Look at
$w_4$, whose true value is exactly $0.0$: at $\lambda = 10^{-4}$ ridge still gives it
$0.81$. **Ridge shrinks but does not select** - the $\ell_2$ penalty has no kink at the
origin, so nothing pins a coefficient to exactly zero. That is day 6's job.

**Middle.** $\|w\|$ is finite everywhere, diverging only as $\lambda \to 0$ where we are
back to the unattainable MLE. The penalty has not made the data less separable; it has made
running away cost something.

**Right - the bonus.** $\kappa(H)$ at the solution falls from 106 to 1.02 as $\lambda$
grows, and the Newton iteration count tracks it. Adding $\lambda I$ raises every eigenvalue
by $\lambda$, bounding $\kappa \le (L + \lambda)/\lambda$ *regardless of the data*. The
same one-line change that fixed the statistics also guarantees Cholesky can never raise
here again.

That is the bridge to tomorrow: if $H + \lambda I$ is always factorisable and $\lambda$
interpolates between Newton and gradient descent, the only thing left to invent is a rule
for choosing $\lambda$ at each step.

---

## Checkpoint

Everything from day 1 to day 4 should be green before you leave, and
`mypy --strict` must be clean. A red type check counts as a failure.

In [ ]:
check("-m", "day1 or day2 or day3 or day4")

In [ ]:
r = subprocess.run([sys.executable, "-m", "mypy"], cwd=OPTLAB,
                   capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip())

## Checkpoint

1. Your `CholeskySolver` raises on a Hessian. A colleague proposes catching it and using
   $-g$ instead, "so the optimizer never crashes". Using Figure 2, say what information
   that throws away and what you would do instead.
2. Newton took 6 iterations and gradient descent 568. Using your own Figure 3, at what
   cost ratio does Newton stop being worth it on this problem, and roughly which $d$ is
   that on your machine? Would doubling $n$ move that $d$ up or down?
3. In Sec. 5, both $f$ and $\|\nabla f\|$ went to zero while the answer got worse. Write down
   a stopping condition that would have caught it.
4. You add `L2(lam)` and your fit barely changes, whatever `lam` is. Name two things that
   could be true, and one cell from this notebook that distinguishes them.
5. Predict, then check: what does Sec. 1 print if you replace `A` with a matrix having one
   negative eigenvalue - for example `np.array([[4.0, 1.0], [1.0, -3.0]])`?

**Before day 5.** Labwork 5 needs `ILeastSquaresProblem` with its `residuals` and
`jacobian`, plus `GaussNewton` and `LevenbergMarquardt`. Note that
`ILeastSquaresProblem` is deliberately **not** an `IObjective` - it exposes residuals and a
Jacobian, not a scalar and a gradient. Lecture 5 explains why that separation is an
interface-segregation decision and not an oversight.

---

## Before the debrief

Be able to answer:

1. Why not invert `H`? Give both the flop count and the numerical reason.
2. What does "Cholesky failed" mean **geometrically**, and what does it mean
   **statistically**? They are the same sentence in two languages.
3. Your `CholeskySolver` is about to be reused twice tomorrow without modification. Which
   design decision made that possible?
4. `check_hessian` compares your Hessian against the *numerical* Jacobian of your gradient,
   so it inherits the finite-difference floor twice over. A Hessian is the Jacobian of a
   gradient, and `jacobian_forward` from `optlab.autodiff` takes Jacobians exactly - so how
   would you check `GLMLoss.hessian` to machine precision? (Write the analytic gradient
   `X^T (mu(z) - y) / n` over `Dual`, and take its Jacobian. Tomorrow's Exercise 1 uses the
   same function for a different purpose.)